In [ ]:
"""
sandbox_lite.ipynb

A sandbox to develop a lighter version of the code.

Author: Stellina X. Ao
Created: 2026-07-07
Last Modified: 2026-07-07
Python Version: 3.11.14
"""


import scienceplots  # noqa: F401
import shutup
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

In [ ]:
subj_id = "MR82"
sess_id = "20251027_152036"  # "20251028_140930" # "20251027_152036"

In [ ]:
"""--------------------------------------------"""
# add in model parameters to encoder, look at cvr2
# cvr2 with strategy model params
# reward prediction error (MF), q-learner, need to get q-value
"""---------------------------------------------"""
# build in interaction terms and see what pops out in the cvr2/dr2 plots
"""---------------------------------------------"""
# add movement over time
"""---------------------------------------------"""
# cvr2 across time
# --> when does encoding emerge across time?
"""---------------------------------------------"""
# aggregate across sessions
"""---------------------------------------------"""
# check for temporal autocorrelations by ensuring no encoding after shuffling trial data.
# incorporate as a sanity check to pass for all sessions

# > check the fits for different regularization constants
# define responsive

# one regressor
"""--------------------------------------------"""

## init

In [ ]:
from sg.models import Encoder, StrategyEncoder

encoder = Encoder(subj_id, sess_id, norm=True)
encoder.verify()

encoder_mb = StrategyEncoder(subj_id, sess_id, norm=True, strategy_filter="mb")
encoder_mf = StrategyEncoder(subj_id, sess_id, norm=True, strategy_filter="mf")

encoder_mb.verify()
encoder_mb.fit_encoder()

encoder_mf.verify()
encoder_mf.fit_encoder()

In [ ]:
encoder.encoder_predict()

In [ ]:
quot = encoder.robs / encoder.robs_predict["encoder"]

In [ ]:
from sklearn.neural_network import MLPRegressor as NN

X = encoder.robs
y = encoder.robs - encoder.robs_predict["encoder"]

sg_a_estimator = NN(hidden_layer_sizes=[1]).fit(X, y)

In [ ]:
v = sg_a_estimator.coefs_[-1].flatten()

In [ ]:
W1 = sg_a_estimator.coefs_[0]
b1 = sg_a_estimator.intercepts_[0]

m = X @ W1 + b1

vm = v * m

In [ ]:
robs_predict_sg = vm + encoder.robs_predict["encoder"]

In [ ]:
from squiggs.renderers import FitRenderer
from squiggs.neuron_viewer import NeuronViewer
from utils.paths import FIGURES_DIR

reg = "DLS"

r = FitRenderer(
    y=encoder.robs[:, encoder.reg_idxs[reg]] * (1000 / encoder.binwidth_ms),
    yhat=robs_predict_sg[:, encoder.reg_idxs[reg]] * (1000 / encoder.binwidth_ms),
    ylabel="Firing Rate (Hz)",
    rsquared=encoder.scores["encoder"][encoder.reg_idxs[reg]],
    mode="lite",
)

NeuronViewer(num_units=encoder.psths[reg].shape[0], render_func=r, fig_dir=FIGURES_DIR)

In [ ]:
plt.figure()
plt.plot(y)
plt.show()

In [ ]:
from scipy.stats import pearsonr as r

corr = {
    tv: r(m.flatten(), encoder.trial_data[tv].values).statistic
    for tv in encoder.tv_keys
}
corr

In [ ]:
encoder.trial_data["response"].values.shape

In [ ]:
encoder.view_peths()

In [ ]:
encoder_mf.view_fits()